In [5]:
from google.colab import files
uploaded = files.upload()

Saving paket_a_transaksi_pos_2024_2025.csv to paket_a_transaksi_pos_2024_2025.csv


In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')

In [8]:
df['harga_satuan'] = df['harga_satuan'].astype(str).str.replace(',', '.').astype(float)

In [9]:
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

In [10]:
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

In [11]:
kolom_target = ['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']
indeks_kolom = [df.columns.get_loc(c) for c in kolom_target]
subset_iloc = df.iloc[0:5, indeks_kolom]

In [12]:
print("Hasil Task 1 - Subset Slicing (.iloc):")
display(subset_iloc)

Hasil Task 1 - Subset Slicing (.iloc):


,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


In [13]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']

In [14]:
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)

In [15]:
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

In [16]:
print("Hasil Task 2 - Sampel Feature Engineering:")
display(df[['id_transaksi', 'Total_Kotor', 'diskon_persen', 'Nominal_Diskon', 'Total_Bersih']].head())

Hasil Task 2 - Sampel Feature Engineering:


,id_transaksi,Total_Kotor,diskon_persen,Nominal_Diskon,Total_Bersih
0,TRX0241065,52000.0,0,0.0,57720.0
1,TRX0240510,21000.0,0,0.0,23310.0
2,TRX0244483,22500.0,0,0.0,24975.0
3,TRX0242532,13000.0,0,0.0,14430.0
4,TRX0243509,6000.0,0,0.0,6660.0


In [17]:
f_multi = df[(df['status'] == 'SELESAI') &
             (df['kode_gerai'].isin(['G05', 'G09'])) &
             (df['Total_Bersih'] > 100000)]

In [18]:
print(f"Jumlah baris data yang memenuhi kriteria multi-kondisi: {len(f_multi)}")
display(f_multi.head())

Jumlah baris data yang memenuhi kriteria multi-kondisi: 139


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500.0,10,SELESAI,190000.0,19000.0,189810.0
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,30500.0,0,SELESAI,122000.0,0.0,135420.0
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,28500.0,0,SELESAI,114000.0,0.0,126540.0
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,46500.0,0,SELESAI,139500.0,0.0,154845.0
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,20500.0,0,SELESAI,123000.0,0.0,136530.0


In [19]:
f_query = df.query("kode_gerai == 'G10' and jumlah >= 5")
print(f"\nJumlah baris data gerai G10 (jumlah >= 5): {len(f_query)}")
display(f_query.head())


Jumlah baris data gerai G10 (jumlah >= 5): 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0


In [20]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])

In [21]:
print("Sampel Hasil Multi-Column Sorting (A-Z Gerai, Total Bersih Descending):")
display(df_sorted[['id_transaksi', 'kode_gerai', 'Total_Bersih']].head())

Sampel Hasil Multi-Column Sorting (A-Z Gerai, Total Bersih Descending):


,id_transaksi,kode_gerai,Total_Bersih
244,TRX0243257,G01,42020160.0
2956,TRX0241367,G01,672660.0
3588,TRX0240608,G01,626040.0
2937,TRX0240631,G01,559440.0
5,TRX0240658,G01,545454.0


In [22]:
top_5_transaksi = df.nlargest(5, 'Total_Bersih')

In [23]:
print("\nTask 4.2 - Top 5 Transaksi dengan Total Bersih Tertinggi:")
display(top_5_transaksi[['id_transaksi', 'kode_gerai', 'jumlah', 'Total_Bersih']])


Task 4.2 - Top 5 Transaksi dengan Total Bersih Tertinggi:


,id_transaksi,kode_gerai,jumlah,Total_Bersih
994,TRX0241002,G11,2093,100363536.0
4985,TRX0244221,G09,1301,66934498.5
467,TRX0243696,G06,892,53466480.0
3393,TRX0241029,G07,1983,45563391.0
244,TRX0243257,G01,1456,42020160.0


In [24]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai', as_index=False).agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

print("Task 5.1 - Laporan Agregasi Kinerja Gerai:")
display(laporan_gerai)

Task 5.1 - Laporan Agregasi Kinerja Gerai:


,kode_gerai,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
10,G11,398,3170,1.253536e+08,314958.82
5,G06,383,3103,9.113936e+07,237961.77
8,G09,379,3027,9.019606e+07,237984.32
6,G07,384,4284,8.036181e+07,209275.55
0,G01,450,2635,7.077959e+07,157287.97
2,G03,372,3274,5.980970e+07,160778.77
9,G10,401,1742,5.892248e+07,146938.85
7,G08,408,1700,3.550256e+07,87016.08
11,G12,406,1010,2.464483e+07,60701.55
3,G04,399,1042,2.453577e+07,61493.15


In [25]:
total_omzet_per_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = (df['Total_Bersih'] / total_omzet_per_gerai) * 100

print("\nTask 5.2 - Sampel DataFrame dengan Kolom Kontribusi Gerai (%):")
display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head())


Task 5.2 - Sampel DataFrame dengan Kolom Kontribusi Gerai (%):


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,57720.0,0.063031
1,TRX0240510,G09,23310.0,0.025455
2,TRX0244483,G11,24975.0,0.018364
3,TRX0242532,G05,14430.0,0.060124
4,TRX0243509,G05,6660.0,0.027750
